# Time Series and Performance

Time series data is everywhere in data science: stock prices, sensor readings, web traffic, and more. Pandas has excellent support for working with datetime data. This notebook also covers performance optimization—critical knowledge for working with large datasets.

## Learning Objectives

- Convert to and work with DatetimeIndex
- Use the `.dt` accessor for date/time components
- Resample time series to different frequencies
- Apply rolling window calculations
- Understand why `.iterrows()` is slow and what to use instead
- Optimize memory usage with dtype downcasting

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import time

In [ ]:
# Load bike share data
bikes = pd.read_csv('../../data/berlin_bikes.csv')
print(f"Shape: {bikes.shape}")
bikes.head()

## Working with Datetime

In [ ]:
# Convert string to datetime
bikes['start_time'] = pd.to_datetime(bikes['start_time'])
bikes['end_time'] = pd.to_datetime(bikes['end_time'])

print("Dtypes after conversion:")
print(bikes[['start_time', 'end_time']].dtypes)

In [ ]:
# Set datetime as index
bikes_ts = bikes.set_index('start_time').sort_index()
print("DatetimeIndex:")
print(bikes_ts.index[:5])

## The .dt Accessor

Access date/time components from a datetime column.

In [ ]:
# Extract components
bikes['year'] = bikes['start_time'].dt.year
bikes['month'] = bikes['start_time'].dt.month
bikes['day'] = bikes['start_time'].dt.day
bikes['hour'] = bikes['start_time'].dt.hour
bikes['dayofweek'] = bikes['start_time'].dt.dayofweek  # 0=Monday
bikes['day_name'] = bikes['start_time'].dt.day_name()

print("Extracted datetime components:")
bikes[['start_time', 'year', 'month', 'day', 'hour', 'dayofweek', 'day_name']].head(10)

In [ ]:
# Weekend flag
bikes['is_weekend'] = bikes['start_time'].dt.dayofweek >= 5

print("Weekend distribution:")
print(bikes['is_weekend'].value_counts(normalize=True))

## Resampling

Resample converts time series data from one frequency to another.

In [ ]:
# Daily trip counts
daily_trips = bikes_ts.resample('D').size()
print("Daily trips:")
daily_trips.head(10)

In [ ]:
# Weekly average duration
weekly_duration = bikes_ts['duration'].resample('W').mean()
print("Weekly average duration:")
weekly_duration.head()

In [ ]:
# Multiple aggregations
monthly_stats = bikes_ts['duration'].resample('M').agg(['count', 'mean', 'median', 'std'])
print("Monthly statistics:")
monthly_stats.head()

In [ ]:
# Visualize daily trips
fig, ax = plt.subplots(figsize=(14, 5))
daily_trips.plot(ax=ax, alpha=0.7)
ax.set_title('Daily Bike Trips')
ax.set_xlabel('Date')
ax.set_ylabel('Number of Trips')
plt.tight_layout()
plt.show()

## Rolling Windows

In [ ]:
# 7-day rolling mean
daily_trips_df = daily_trips.to_frame('trips')
daily_trips_df['rolling_7d'] = daily_trips_df['trips'].rolling(window=7).mean()
daily_trips_df['rolling_30d'] = daily_trips_df['trips'].rolling(window=30).mean()

print("Rolling averages:")
daily_trips_df.head(15)

In [ ]:
# Visualize with rolling average
fig, ax = plt.subplots(figsize=(14, 6))

ax.plot(daily_trips_df.index, daily_trips_df['trips'], alpha=0.3, label='Daily')
ax.plot(daily_trips_df.index, daily_trips_df['rolling_7d'], label='7-day MA', linewidth=2)
ax.plot(daily_trips_df.index, daily_trips_df['rolling_30d'], label='30-day MA', linewidth=2)

ax.set_title('Daily Bike Trips with Moving Averages')
ax.set_xlabel('Date')
ax.set_ylabel('Number of Trips')
ax.legend()
plt.tight_layout()
plt.show()

## Performance: Why iterrows() is Almost Always Wrong

Looping over rows is a common beginner mistake. Let's see why it's so slow.

In [ ]:
# Create test data
n = 50000
test_df = pd.DataFrame({
    'A': np.random.randn(n),
    'B': np.random.randn(n)
})

print(f"Testing with {n:,} rows")

In [ ]:
# Method 1: iterrows() - SLOW!
start = time.perf_counter()
result1 = []
for idx, row in test_df.iterrows():
    result1.append(row['A'] + row['B'])
iterrows_time = time.perf_counter() - start
print(f"iterrows(): {iterrows_time:.3f}s")

In [ ]:
# Method 2: itertuples() - Better but still slow
start = time.perf_counter()
result2 = []
for row in test_df.itertuples():
    result2.append(row.A + row.B)
itertuples_time = time.perf_counter() - start
print(f"itertuples(): {itertuples_time:.3f}s")

In [ ]:
# Method 3: apply() - Still not great
start = time.perf_counter()
result3 = test_df.apply(lambda row: row['A'] + row['B'], axis=1)
apply_time = time.perf_counter() - start
print(f"apply(): {apply_time:.3f}s")

In [ ]:
# Method 4: Vectorized - FAST!
start = time.perf_counter()
result4 = test_df['A'] + test_df['B']
vec_time = time.perf_counter() - start
print(f"vectorized: {vec_time:.5f}s")

In [ ]:
# Visualize the comparison
methods = ['iterrows()', 'itertuples()', 'apply()', 'vectorized']
times = [iterrows_time, itertuples_time, apply_time, vec_time]

fig, ax = plt.subplots(figsize=(10, 6))
colors = ['#e74c3c', '#e67e22', '#f1c40f', '#2ecc71']
bars = ax.bar(methods, times, color=colors)

ax.set_ylabel('Time (seconds)')
ax.set_title(f'Row Operation Performance Comparison ({n:,} rows)')

# Add value labels
for bar, t in zip(bars, times):
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height(),
            f'{t:.3f}s', ha='center', va='bottom')

# Add speedup annotation
speedup = iterrows_time / vec_time
ax.text(0.5, 0.95, f'Vectorized is {speedup:.0f}x faster than iterrows()',
        transform=ax.transAxes, ha='center', fontsize=12, fontweight='bold')

ax.set_yscale('log')
plt.tight_layout()
plt.show()

> **Gotcha: When you think you need iterrows()**
> 
> Before reaching for a loop, ask: "Can I express this with vectorized operations?"
> - Arithmetic: `df['A'] + df['B']`
> - Conditionals: `np.where()` or `np.select()`
> - String operations: `.str` accessor
> - Group operations: `.groupby().transform()`

## Memory Optimization

In [ ]:
# Check memory usage
print("Memory usage before optimization:")
print(bikes.memory_usage(deep=True))
print(f"\nTotal: {bikes.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

In [ ]:
# Downcast numeric types
bikes_opt = bikes.copy()

# Integer columns
for col in bikes_opt.select_dtypes(include=['int64']).columns:
    bikes_opt[col] = pd.to_numeric(bikes_opt[col], downcast='integer')

# Float columns
for col in bikes_opt.select_dtypes(include=['float64']).columns:
    bikes_opt[col] = pd.to_numeric(bikes_opt[col], downcast='float')

print("Integer columns after downcast:")
print(bikes_opt.select_dtypes(include=['integer']).dtypes)

In [ ]:
# Convert low-cardinality strings to category
for col in ['start_station', 'end_station', 'user_type']:
    if col in bikes_opt.columns:
        bikes_opt[col] = bikes_opt[col].astype('category')

print("\nMemory usage after optimization:")
print(f"Total: {bikes_opt.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

original_mem = bikes.memory_usage(deep=True).sum()
optimized_mem = bikes_opt.memory_usage(deep=True).sum()
print(f"\nReduction: {(1 - optimized_mem/original_mem)*100:.1f}%")

## Method Chaining Style

A professional Pandas pattern for readable data transformations.

In [ ]:
# Example: Complete analysis pipeline
hourly_pattern = (
    bikes
    .assign(
        hour=lambda df: df['start_time'].dt.hour,
        is_weekday=lambda df: df['start_time'].dt.dayofweek < 5
    )
    .groupby(['hour', 'is_weekday'])
    .agg(
        trip_count=('duration', 'count'),
        avg_duration=('duration', 'mean')
    )
    .reset_index()
    .round(2)
)

print("Hourly pattern:")
hourly_pattern.head(10)

In [ ]:
# Visualize weekday vs weekend patterns
fig, ax = plt.subplots(figsize=(12, 6))

weekday = hourly_pattern[hourly_pattern['is_weekday'] == True]
weekend = hourly_pattern[hourly_pattern['is_weekday'] == False]

ax.plot(weekday['hour'], weekday['trip_count'], 'b-o', label='Weekday', linewidth=2)
ax.plot(weekend['hour'], weekend['trip_count'], 'r-s', label='Weekend', linewidth=2)

ax.set_xlabel('Hour of Day')
ax.set_ylabel('Number of Trips')
ax.set_title('Bike Usage Pattern: Weekday vs Weekend')
ax.set_xticks(range(24))
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Exercises

### Exercise 1: Date Extraction

Using the bike data, create columns for:
1. Quarter (Q1, Q2, Q3, Q4)
2. Whether it's morning (6-12), afternoon (12-18), or evening (18-24) / night (0-6)
3. Trip duration in minutes (from the duration column, assuming it's in seconds)

In [ ]:
# Extract date components and create time of day
# YOUR CODE HERE

### Exercise 2: Resampling Analysis

Resample the bike data to get:
1. Hourly trip counts
2. Weekly total duration
3. Monthly average duration with standard deviation

In [ ]:
# Perform resampling operations
# YOUR CODE HERE

### Exercise 3: Rolling Analysis

Calculate and plot:
1. 7-day rolling standard deviation of daily trip counts
2. Identify the dates with the highest volatility (highest rolling std)

In [ ]:
# Rolling analysis
# YOUR CODE HERE

### Exercise 4: Memory Optimization

Load the Titanic dataset and optimize its memory usage by:
1. Downcasting numeric columns
2. Converting appropriate string columns to category
3. Report the memory savings percentage

In [ ]:
# Optimize Titanic dataset memory
# YOUR CODE HERE